# 08 - From NNRTI patterns to DTG resistance

This notebook tests a specific longitudinal hypothesis: did places with earlier NNRTI-based resistance later show dolutegravir (DTG) resistance, and did places with little earlier NNRTI resistance later show DTG resistance? It requires comparable geographic observations from at least two time windows. Association is not evidence that NNRTI resistance caused later DTG resistance.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA = Path('data')
candidates = [DATA / 'hivdb_country_resistance_time.csv', DATA / 'current' / 'hivdb_country_resistance_time_current.csv']
path = next((p for p in candidates if p.exists()), None)
required = {'iso3', 'sample_year', 'nnrti_resistance_pct', 'dtg_resistance_pct', 'n_tested'}
if path is None:
    print('No time-indexed country resistance file found.')
    print('The 2015 baseline predates widespread DTG use and cannot answer this question by itself.')
    history = pd.DataFrame()
else:
    history = pd.read_csv(path)
    missing = required - set(history.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    history['sample_year'] = pd.to_numeric(history['sample_year'], errors='coerce')
    history['n_tested'] = pd.to_numeric(history['n_tested'], errors='coerce')
    history = history[history['n_tested'].gt(0)].copy()
    print(f'Loaded {len(history):,} country-time observations')

## Define early and late windows

Choose windows based on the data collection years and document the lag. Require a minimum denominator and comparable ascertainment. If a country was not studied in the early period, it must not be silently treated as having zero NNRTI resistance.

In [ ]:
if not history.empty:
    early_end = 2015
    late_start = 2016
    early = (history.query('sample_year <= @early_end').groupby('iso3')
             .apply(lambda g: pd.Series({'early_nnrti_pct': (g.nnrti_resistance_pct * g.n_tested).sum() / g.n_tested.sum(), 'early_n_tested': g.n_tested.sum()}), include_groups=False).reset_index())
    late = (history.query('sample_year >= @late_start').groupby('iso3')
            .apply(lambda g: pd.Series({'late_dtg_pct': (g.dtg_resistance_pct * g.n_tested).sum() / g.n_tested.sum(), 'late_n_tested': g.n_tested.sum()}), include_groups=False).reset_index())
    comparison = early.merge(late, on='iso3', how='inner').dropna()
    display(comparison.sort_values('late_dtg_pct', ascending=False).head(30))
    fig = px.scatter(comparison, x='early_nnrti_pct', y='late_dtg_pct', hover_name='iso3',
                     labels={'early_nnrti_pct': 'Earlier NNRTI resistance (%)', 'late_dtg_pct': 'Later DTG resistance (%)'},
                     title='Earlier NNRTI pattern versus later DTG resistance')
    fig.show()
else:
    print('No paired early/late estimates are available.')

## What would make the comparison credible?

Use mutation or drug-specific definitions consistently; distinguish transmitted from acquired resistance; stratify or adjust for subtype and ART history; record testing method and denominator; and show uncertainty. A null early NNRTI estimate means missing evidence unless the early denominator is known. The result should be framed as a hypothesis-generating ecological comparison, not an individual-level prediction.